# 03 - Evaluation
Extraction quality vs CUAD v1 expert labels and LLM-judged groundedness of risk explanations. The eval itself runs as the bundle job `eval_job` (serverless); this notebook reads the logged results from Unity Catalog and the MLflow run.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
sql("SELECT clause_type, basis, count(*) docs, count_if(gt_present) gt_present FROM ${catalog}.ml.eval_ground_truth GROUP BY ALL ORDER BY 1")

,clause_type,basis,docs,gt_present
0,assignment_restriction,yes_no,20,16
1,auto_renewal,span_proxy,20,7
2,effective_date,span,20,14
3,exclusivity,yes_no,20,7
4,expiration_date,span,20,17
5,governing_law,span,20,19
6,liability_cap,yes_no,20,11
7,non_compete,yes_no,20,9
8,notice_period_days,span,20,3
9,parties,span,20,20


In [3]:
from export_evidence import latest_eval_run
run_id, exp_id, ts = latest_eval_run(); print('MLflow experiment', exp_id, '| run', run_id, '| at', ts)
run = ws().experiments.get_run(run_id).run
print(run.info.run_name, run.info.status)
pd.DataFrame([(m.key, m.value) for m in run.data.metrics if '.' not in m.key], columns=['metric','value']).sort_values('metric')

MLflow experiment 702680364159558 | run 7afbcff99f564d3aaceed89d9fbc667e | at 2026-11-18T00:14:38.000Z
eval-ai_extract:v2.1-20261118-001122 RunInfoStatus.FINISHED


,metric,value
0,cost_per_1k_docs_usd,119.340000
1,explanation_groundedness/mean,0.633333
2,groundedness,0.633333
3,judge_failures,0.000000
4,judged_explanations,30.000000
5,key_clause_f1,0.775862
6,key_clause_precision,0.865385
7,key_clause_recall,0.703125
8,macro_f1,0.768251
9,micro_f1,0.812721


In [4]:
sql(f"SELECT clause_type, tp, fp, fn, round(precision,3) p, round(recall,3) r, round(f1,3) f1, value_compared, round(exact_match_rate,3) exact, round(fuzzy_match_rate,3) fuzzy FROM {CATALOG}.ml.eval_results WHERE mlflow_run_id = '{run_id}' AND clause_type <> '__summary__' ORDER BY f1")

,clause_type,tp,fp,fn,p,r,f1,value_compared,exact,fuzzy
0,termination_for_convenience,3,3,5,0.5,0.375,0.429,0,NaN,NaN
1,uncapped_liability,4,2,3,0.667,0.571,0.615,0,NaN,NaN
2,expiration_date,10,2,7,0.833,0.588,0.69,10,0.9,0.9
3,exclusivity,6,4,1,0.6,0.857,0.706,0,NaN,NaN
4,non_compete,6,2,3,0.75,0.667,0.706,0,NaN,NaN
5,auto_renewal,4,0,3,1.0,0.571,0.727,0,NaN,NaN
6,liability_cap,7,1,4,0.875,0.636,0.737,0,NaN,NaN
7,notice_period_days,3,2,0,0.6,1.0,0.75,3,1.0,1.0
8,effective_date,13,5,1,0.722,0.929,0.813,13,1.0,1.0
9,renewal_term,7,1,0,0.875,1.0,0.933,6,0.0,0.833


In [5]:
sql(f"SELECT metric_name, round(metric_value,4) value FROM {CATALOG}.ml.eval_results WHERE mlflow_run_id = '{run_id}' AND clause_type = '__summary__' ORDER BY 1")

,metric_name,value
0,cost_per_1k_docs_usd,119.34
1,groundedness,0.6333
2,judge_failures,0.0
3,judge_retried_ok,0.0
4,judged_explanations,30.0
5,key_clause_f1,0.7759
6,key_clause_precision,0.8654
7,key_clause_recall,0.7031
8,macro_f1,0.7683
9,micro_f1,0.8127


In [6]:
sql("SELECT clause_type, gt_answer, predicted_value, exact_match, fuzzy_match FROM ${catalog}.ml.eval_details WHERE clause_type IN ('governing_law','expiration_date','notice_period_days') AND exact_match IS NOT NULL ORDER BY clause_type LIMIT 15")

,clause_type,gt_answer,predicted_value,exact_match,fuzzy_match
0,expiration_date,12/31/22,2022-12-31,true,true
1,expiration_date,12/11/18,2018-12-11,true,true
2,expiration_date,3/29/19,2019-03-29,true,true
3,expiration_date,1/30/03,2003-01-29,false,false
4,expiration_date,1/31/24,2024-01-31,true,true
5,expiration_date,2/1/21,2021-02-01,true,true
6,expiration_date,5/3/18,2018-05-03,true,true
7,expiration_date,2/28/04,2004-02-28,true,true
8,expiration_date,5/31/16,2016-05-31,true,true
9,expiration_date,12/31/28,2028-12-31,true,true


In [7]:
sql("SELECT grounded, count(*) n FROM ${catalog}.ml.eval_groundedness GROUP BY 1")

,grounded,n
0,false,11
1,true,19


In [8]:
sql("SELECT g.rule_id, left(g.explanation, 200) explanation, left(e.rationale, 220) rationale FROM ${catalog}.ml.eval_groundedness e JOIN ${catalog}.curated.gold_risk_flags g USING (flag_id) WHERE NOT e.grounded LIMIT 5")

,rule_id,explanation,rationale
0,R02,This clause triggers the non-compete rule because it explicitly restricts Allied from endorsing or permitting market...,The explanation contains a factual assertion that is not supported by the source snippet. While the explanation corr...
1,R02,"This clause triggers the non-compete rule because it explicitly restricts the Franchisee from owning, managing, or o...",The explanation makes several factual assertions about the contract that are NOT supported by the SOURCE SNIPPET pro...
2,R02,"This clause triggers the rule because it explicitly restricts each Seller from ""directly, or indirectly...own, manag...",The explanation contains factual assertions about the contract that are NOT supported by the SOURCE SNIPPET provided...
3,R05,"This clause triggers the exclusivity rule because it explicitly grants the Reseller ""the exclusive right to commissi...","The explanation makes a factual assertion that is not supported by the SOURCE SNIPPET. Specifically, the explanation..."
4,R02,"This clause triggers the rule because it explicitly restricts Auriemma from ""enter[ing] into any arrangement or agre...","The explanation contains a factual assertion that is not supported by the source snippet. Specifically, the explanat..."


In [9]:
sql("SELECT stage, docs, pages, input_tokens, output_tokens, est_cost_usd, cost_per_1k_docs FROM ${catalog}.curated.gold_cost_metrics ORDER BY stage")

,stage,docs,pages,input_tokens,output_tokens,est_cost_usd,cost_per_1k_docs
0,classify,20,NaN,91793,600,0.0468,2.34
1,explain,13,NaN,6320,3367,0.0232,1.16
2,extract,20,NaN,212067,13912,0.1269,6.35
3,parse,20,438,NaN,NaN,2.19,109.5
4,total,20,438,310180,17879,2.3869,119.34
